losses

In [1]:
%run code/losses.py

decoder_kd

In [2]:
import os
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

from unsloth import FastLanguageModel

import torch
from transformers import Trainer, TrainingArguments
from trl import SFTTrainer, SFTConfig

# from losses import compute_fkl, compute_rkl
from datasets import load_dataset

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [3]:
max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.


class KDTrainer(SFTTrainer):

    def __init__(self, *args, teacher_model=None, if_use_entropy=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.teacher_model = teacher_model
        self.if_use_entropy = if_use_entropy

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        outputs_student = model(**inputs)

        with torch.no_grad():
            teacher_outputs = self.teacher_model(**inputs)

        loss = outputs_student.loss
        logits = outputs_student.logits

        with torch.no_grad():
            teacher_logits = teacher_outputs.logits

        # 如果教师模型和学生模型输出形状不匹配，对学生模型进行padding或对教师模型进行截断
        # print(logits.shape, teacher_logits.shape)
        # print(type(logits), type(teacher_logits))
        # if logits is None or teacher_logits is None:

        kl = 0
        if isinstance(logits, torch.Tensor) and isinstance(teacher_logits, torch.Tensor):
            if logits.shape[-1] != teacher_logits.shape[-1]:
                # gap = teacher_logits.shape[-1] - logits.shape[-1]
                # if gap > 0:
                #     pad_logits = torch.zeros((logits.shape[0], logits.shape[1], gap)).to(logits.device)
                #     logits = torch.cat([logits, pad_logits], dim=-1)

                teacher_logits = teacher_logits[:, :, :logits.shape[-1]]

                labels = inputs['labels']
                # kl = compute_fkl(logits, teacher_logits, labels, padding_id=-100, temp=2.0)
                kl = compute_rkl(logits, teacher_logits, labels, padding_id=-100, temp=2.0)

        if self.if_use_entropy:
            loss_total = 0.5 * kl + 0.5 * loss
        else:
            loss_total = kl

        return (loss_total, outputs_student) if return_outputs else loss_total

In [4]:
# 学生模型
student, _ = FastLanguageModel.from_pretrained(
    # Can select any from the below:
    # "unsloth/Qwen2.5-0.5B", "unsloth/Qwen2.5-1.5B", "unsloth/Qwen2.5-3B"
    # "unsloth/Qwen2.5-14B",  "unsloth/Qwen2.5-32B",  "unsloth/Qwen2.5-72B",
    # And also all Instruct versions and Math. Coding verisons!
    model_name = "unsloth/Qwen2.5-1.5B",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    # token = "hf_...", # use one if using gated models like meta-llama/Llama-2-7b-hf
)

student = FastLanguageModel.get_peft_model(
    student,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

student.print_trainable_parameters()

teacher, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "qwen_teacher_finetune",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

FastLanguageModel.for_inference(teacher)

==((====))==  Unsloth 2026.7.5: Fast Qwen2 patching. Transformers: 5.5.0. vLLM: 0.21.0.
   \\   /|    NVIDIA GeForce RTX 4090. Num GPUs = 1. Max memory: 23.516 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 8.9. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Error while downloading from https://cas-bridge.xethub.hf.co/xet-bridge-us/67a41fb635a7aa0f914011fc/a7030cf2e58dead38199a68a8cd6f6f1a609a6072d7fb38ba5f85b3bb7e21557?Expires=1785003450&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly9jYXMtYnJpZGdlLnhldGh1Yi5oZi5jby94ZXQtYnJpZGdlLXVzLzY3YTQxZmI2MzVhN2FhMGY5MTQwMTFmYy9hNzAzMGNmMmU1OGRlYWQzODE5OWE2OGE4Y2Q2ZjZmMWE2MDlhNjA3MmQ3ZmIzOGJhNWY4NWIzYmI3ZTIxNTU3KiIsIkNvbmRpdGlvbiI6eyJEYXRlTGVzc1RoYW4iOnsiQVdTOkVwb2NoVGltZSI6MTc4NTAwMzQ1MH19fV19&Signature=MEYCIQDWvr6UAasJ6kXJ9GMeaIA1MFzQQXSHi58DKXLu%7EHhYlgIhAPTd8zA3RtLFrjlJL8u4t5ui073UX9hxnpHQcD85-LOp&Key-Pair-Id=K3EPXBYC3CKDRZ&X-Xet-Cas-Uid=62171e3b6a99db28e0b3159d&response-content-type=application%2Fjson&response-content-disposition=inline%3B+filename*%3DUTF-8%27%27tokenizer.json%3B+filename%3D%22tokenizer.json%22%3B&X-Amz-Algorithm=AWS4-HMAC-SHA256&X-Amz-Credential=cas%2F20260725%2Fus-east-1%2Fs3%2Faws4_request&X-Amz-Date=20260725T171730Z&X-Amz-Expires=3600&X-Amz-SignedHeaders=host&X-Amz-Si

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Unsloth 2026.7.5 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820
==((====))==  Unsloth 2026.7.5: Fast Qwen2 patching. Transformers: 5.5.0. vLLM: 0.21.0.
   \\   /|    NVIDIA GeForce RTX 4090. Num GPUs = 1. Max memory: 23.516 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 8.9. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(152064, 3584, padding_idx=151654)
        (layers): ModuleList(
          (0-1): 2 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=3584, out_features=3584, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=3584, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=3584, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear(
 

In [5]:
# 读取数据集
# process dataset
    alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.
    
    ### Instruction:
    {}
    
    ### Input:
    {}
    
    ### Response:
    {}"""

EOS_TOKEN = tokenizer.eos_token # Must add EOS_TOKEN
def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input, output in zip(instructions, inputs, outputs):
        # Must add EOS_TOKEN, otherwise your generation will go on forever!
        text = alpaca_prompt.format(instruction, input, output) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts, }
pass


from datasets import load_dataset
dataset = load_dataset("yahma/alpaca-cleaned", split = "train[:2000]")
dataset = dataset.map(formatting_prompts_func, batched = True,)

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [7]:
# 训练参数
args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=10,
    do_train=True,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=16,
    logging_steps=10,

    save_strategy='epoch',
    save_total_limit=10,
    bf16=True,
    learning_rate=0.0005,
    lr_scheduler_type='constant',
    optim="adamw_torch_fused",
)


trainer = KDTrainer(
    model=student,
    teacher_model=teacher,

    if_use_entropy=True,
    processing_class=tokenizer,
    train_dataset=dataset,
    dataset_text_field = "text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,  # Can make training 5x faster for short sequences.
    args=args,
)


import sys
import trl.trainer.sft_config
import trl.trainer.sft_trainer

# 获取 trainer.args 和 trainer 在内存中的真实类
real_config_cls = type(trainer.args)
real_trainer_cls = type(trainer)

# 获取这些真实类所在的真实模块对象
real_config_module = sys.modules[real_config_cls.__module__]
real_trainer_module = sys.modules[real_trainer_cls.__module__]

# 强制将 sys.modules 中的标准路径指向这些真实模块
sys.modules['trl.trainer.sft_config'] = real_config_module
sys.modules['trl.trainer.sft_trainer'] = real_trainer_module

# 确保模块内的属性也指向正确的类
real_config_module.SFTConfig = real_config_cls
real_trainer_module.SFTTrainer = real_trainer_cls



# 如果是初次训练resume_from_checkpoint为false，接着checkpoint继续训练，为True
trainer.train(resume_from_checkpoint=False)

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,000 | Num Epochs = 10 | Total steps = 630
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 16
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 16 x 1) = 32
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)
/opt/venv/lib/python3.12/site-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/opt/venv/lib/python3.12/site-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the

Step,Training Loss
10,7.564360
20,7.842282
30,7.792761
40,7.635045
50,7.842752
60,7.838995
70,6.984671
80,6.841795
90,6.390778
100,6.450512


Unsloth: Restored added_tokens_decoder metadata in ./results/checkpoint-63/tokenizer_config.json.
/opt/venv/lib/python3.12/site-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/opt/venv/lib/python3.12/site-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/opt/venv/lib/python3.12/site-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionM

TrainOutput(global_step=630, training_loss=3.245659543597509, metrics={'train_runtime': 3325.4583, 'train_samples_per_second': 6.014, 'train_steps_per_second': 0.189, 'total_flos': 4.134106467276595e+16, 'train_loss': 3.245659543597509, 'epoch': 10.0})

计算与教师模型的KL散度

In [3]:
%run code/losses

In [5]:
import os
# os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

from unsloth import FastLanguageModel
import torch
from transformers import TrainingArguments
from trl import SFTTrainer
import warnings
import transformers
from transformers.trainer_callback import ProgressCallback

# from losses import compute_fkl, compute_rkl
from datasets import load_from_disk

# 强行关闭 HuggingFace 各种警告
transformers.logging.set_verbosity_error()
warnings.filterwarnings("ignore")

max_seq_length = 2048
load_in_4bit = True
max_new_tokens = 128
temperature = 2.0

# ---------------------------------------------------------
# 1. Trainer (迫使模型自行生成，计算 On-Policy KL)
# ---------------------------------------------------------
class OPDTrainer(SFTTrainer):
    def __init__(self, *args, teacher_model=None, max_new_tokens=128, temp=2.0, **kwargs):
        super().__init__(*args, **kwargs)
        self.teacher_model = teacher_model
        self.max_new_tokens = max_new_tokens
        self.temp = temp
        self.tokenizer = kwargs.get("processing_class", kwargs.get("tokenizer"))

    def _generate_on_policy(self, model, input_ids, attention_mask):
        was_training = model.training
        model.eval()
        with torch.no_grad():
            generated_ids = model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_new_tokens=self.max_new_tokens,
                do_sample=True,
                top_p=0.9,
                temperature=0.8,
                pad_token_id=self.tokenizer.eos_token_id,
                eos_token_id=self.tokenizer.eos_token_id,
                use_cache=True,
            )
        if was_training:
            model.train()
        return generated_ids

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        prompt_input_ids = inputs["input_ids"]
        prompt_attention_mask = inputs.get("attention_mask", prompt_input_ids.ne(self.tokenizer.pad_token_id).long())
        prompt_lengths = prompt_attention_mask.sum(dim=1)

        generated_ids = self._generate_on_policy(model, prompt_input_ids, prompt_attention_mask)
        generated_attention_mask = generated_ids.ne(self.tokenizer.pad_token_id).long()

        labels = generated_ids.clone()
        for row_idx, prompt_len in enumerate(prompt_lengths):
            labels[row_idx, :prompt_len] = -100
        labels = labels.masked_fill(generated_attention_mask.eq(0), -100)

        outputs_student = model(
            input_ids=generated_ids,
            attention_mask=generated_attention_mask,
            return_dict=True
        )
        logits = outputs_student.logits

        with torch.no_grad():
            teacher_outputs = self.teacher_model(
                input_ids=generated_ids,
                attention_mask=generated_attention_mask,
                return_dict=True
            )
            teacher_logits = teacher_outputs.logits

        kl = 0
        if isinstance(logits, torch.Tensor) and isinstance(teacher_logits, torch.Tensor):
            if logits.shape[-1] != teacher_logits.shape[-1]:
                teacher_logits = teacher_logits[:, :, :logits.shape[-1]]
            kl = compute_rkl(logits, teacher_logits, labels, padding_id=-100, temp=self.temp)
            valid_tokens = labels.ne(-100).sum().clamp_min(1)
            kl = kl / valid_tokens

        loss_total = kl
        return (loss_total, outputs_student) if return_outputs else loss_total


# ---------------------------------------------------------
# 2. 加载模型与数据
# ---------------------------------------------------------
print("1. 正在加载【Off-Policy】训练完成的 Student 模型...")
student, tokenizer = FastLanguageModel.from_pretrained(
    model_name="./results/checkpoint-630", # <--- 指向离线蒸馏权重
    max_seq_length=max_seq_length,
    load_in_4bit=load_in_4bit,
)
FastLanguageModel.for_inference(student)
student.eval()

print("2. 正在加载 Teacher 模型 (qwen_teacher_finetune)...")
teacher, _ = FastLanguageModel.from_pretrained(
    model_name="qwen_teacher_finetune",
    max_seq_length=max_seq_length,
    load_in_4bit=load_in_4bit,
)
FastLanguageModel.for_inference(teacher)
teacher.eval()

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("3. 加载测试集...")
test_dataset = load_from_disk("./data_splits/data_test")
print(f" 测试集加载成功，共 {len(test_dataset)} 条数据")

# ---------------------------------------------------------
# 3. 启动评估
# ---------------------------------------------------------
args = TrainingArguments(
    output_dir='./eval_results_offpolicy',
    per_device_eval_batch_size=2,
    report_to="none"
)

trainer = OPDTrainer(
    model=student,
    teacher_model=teacher,
    processing_class=tokenizer,
    train_dataset=test_dataset,   
    eval_dataset=test_dataset,    
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    args=args,
    max_new_tokens=max_new_tokens,
    temp=temperature,
)

print("\n 准备在测试集上计算 KL 散度...")

# 动态移除导致崩溃的 HTML 进度条，换成纯文本 tqdm
callbacks_to_remove = []
for callback in trainer.callback_handler.callbacks:
    if "NotebookProgressCallback" in str(type(callback)):
        callbacks_to_remove.append(callback)
for callback in callbacks_to_remove:
    trainer.remove_callback(callback)
trainer.add_callback(ProgressCallback)

metrics = trainer.evaluate()

print("\n" + "="*60)
print(f" 最终评估结果 (Off-Policy 的 Test Set KL Divergence): {metrics['eval_loss']:.4f}")
print("="*60)

1. 正在加载【Off-Policy】训练完成的 Student 模型...
==((====))==  Unsloth 2026.8.2: Fast Qwen2 patching. Transformers: 5.5.0. vLLM: 0.21.0.
   \\   /|    NVIDIA GeForce RTX 5090. Num GPUs = 1. Max memory: 31.357 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 12.0. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

2. 正在加载 Teacher 模型 (qwen_teacher_finetune)...
==((====))==  Unsloth 2026.8.2: Fast Qwen2 patching. Transformers: 5.5.0. vLLM: 0.21.0.
   \\   /|    NVIDIA GeForce RTX 5090. Num GPUs = 1. Max memory: 31.357 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 12.0. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

3. 加载测试集...
 测试集加载成功，共 200 条数据


Unsloth: Tokenizing ["text"] (num_proc=20):   0%|          | 0/200 [00:00<?, ? examples/s]


 准备在测试集上计算 KL 散度...


  0%|          | 0/100 [00:00<?, ?it/s]

{'eval_loss': '1.221', 'eval_model_preparation_time': '0.0264', 'eval_runtime': '499', 'eval_samples_per_second': '0.401', 'eval_steps_per_second': '0.2', 'epoch': 0}

 最终评估结果 (Off-Policy 的 Test Set KL Divergence): 1.2210
